In [ ]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)


# 05 EDA 探索分析 (core.eda)

基于 hscredit.xlsx 真实信贷数据，演示全套 EDA 分析流程。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import hscredit as hsc
from hscredit import (init_setting,
    data_info, missing_analysis, feature_summary, numeric_summary,
    data_quality_report,
    target_distribution, bad_rate_overall, bad_rate_by_dimension,
    bad_rate_trend, bad_rate_by_bins, sample_distribution,
    iv_analysis, batch_iv_analysis, woe_analysis, binning_bad_rate,
    correlation_matrix, high_correlation_pairs, vif_analysis,
    psi_analysis, batch_psi_analysis,
    vintage_analysis, eda_summary,
)
from sklearn.model_selection import train_test_split
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D')
df['loan_month'] = df['loan_date'].dt.to_period('M').astype(str)
target = 'FPD15'
features = [c for c in df.columns if c not in ['MOB1','MOB2','loan_date','loan_month','FPD15','SFPD15']]
print('shape:', df.shape, ' bad_rate:', round(df[target].mean(), 4))

## 1. 数据基本信息

In [ ]:
print(data_info(df))

## 2. 缺失率分析

In [ ]:
miss = missing_analysis(df[features])
print(miss.sort_values('缺失率', ascending=False).head(10))

## 3. 数据质量报告

In [ ]:
data_quality_report(df[features]).head(10)

## 4. 目标变量分析

In [ ]:
print('整体坏率:', bad_rate_overall(df, target))
print()
trend = bad_rate_trend(df, target, date_col='loan_date', freq='M')
print('月度坏率趋势:')
print(trend)

In [ ]:
bad_rate_by_bins(df, feature='bj_qy24', target=target, n_bins=8)

In [ ]:
sample_distribution(df, date_col='loan_date', target=target, freq='M')

## 5. 特征 IV 分析

In [ ]:
iv_res = iv_analysis(df, feature='bj_qy24', target=target, n_bins=8)
print('bj_qy24 IV:', iv_res['IV值'])
iv_res['分箱明细'][['分箱标签','样本总数','坏样本率','分档WOE值','分档IV值']]

In [ ]:
num_feats = df[features].select_dtypes('number').columns.tolist()
batch_iv = batch_iv_analysis(df, features=num_feats, target=target)
batch_iv.sort_values('IV值', ascending=False).head(15)

In [ ]:
woe_res = woe_analysis(df, feature='overdue_loan_m1_count_12m', target=target, n_bins=6)
woe_res['WOE明细']

## 6. 相关性分析

In [ ]:
top_feats = batch_iv.sort_values('IV值', ascending=False).head(10).index.tolist()
high_corr = high_correlation_pairs(df[top_feats].fillna(-999), threshold=0.7)
print('高相关特征对:')
print(high_corr)

In [ ]:
vif_df = vif_analysis(df[top_feats[:8]].fillna(-999))
print('VIF分析:')
print(vif_df)

## 7. 稳定性分析

In [ ]:
df_tr, df_te = train_test_split(df, test_size=0.3, random_state=42)
psi_res = psi_analysis(df_tr, df_te, feature='bj_qy24')
print('bj_qy24 PSI:', psi_res['PSI值'], ' 稳定性:', psi_res['稳定性'])
psi_res['分箱明细']

In [ ]:
months = sorted(df['loan_month'].unique())
batch_psi_df = batch_psi_analysis(df, features=top_feats[:5],
    date_col='loan_month', base_period=months[0], compare_periods=months[1:4])
batch_psi_df

## 8. Vintage 分析

In [ ]:
df_vint = df[['loan_month','MOB1','MOB2','FPD15']].copy()
df_vint['mob'] = df['MOB1']
df_vint = df_vint[df_vint['mob'].notna() & (df_vint['mob'] > 0)]
if len(df_vint) > 0:
    vint_res = vintage_analysis(df_vint, cohort_col='loan_month', mob_col='mob', target='FPD15')
    print('Vintage分析:')
    print(vint_res.head())

## 9. 综合 EDA 报告

In [ ]:
summary = eda_summary(df, target=target, features=top_feats[:6])
for k, v in summary.items():
    print(k, ':', type(v).__name__)